In [ ]:
import os

import json
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

from dynamics_analysis_with_koopman.config.configs import PROJECT_ROOT
from dynamics_analysis_with_koopman.utils.checkpoint_registry import (
    checkpoint_path_for,
    filter_eigenvalues_by_residual,
)

# selection citerion for eigenvalues
KEEP_N = 25
TEMPORAL_RESOLUTION = 0.2

# Where this notebook's figures are stored, and in which format -- change
# FIGURE_FORMAT alone (e.g. to 'eps' or 'png') to switch every plot below.
PLOTS_DIR = PROJECT_ROOT / 'papers' / 'clustering' / 'eigenvalues'
FIGURE_FORMAT = 'svg'
os.makedirs(PLOTS_DIR, exist_ok=True)


def save_figure(name, fig=None):
    (fig or plt).savefig(os.path.join(PLOTS_DIR, f'{name}.{FIGURE_FORMAT}'), bbox_inches='tight')


# formatting for plots
SMALL_SIZE = 16
MEDIUM_SIZE = 18

# importing from dynamics_analysis_with_koopman.plots runs setup_seaborn_style()
# reset to plain matplotlib defaults so this notebook's own plots
plt.rcdefaults()
plt.rc('axes', titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)  # legend fontsize


## helper function
def _get_eigenvalues_from_csv(experiment, with_residuals=False):
    checkpoint_path = checkpoint_path_for(experiment=experiment, registry=registry)
    path = os.path.join(checkpoint_path, 'data', 'eigenvalues.csv')

    eigenvalues = np.genfromtxt(path, delimiter=',', skip_header=0, dtype=None)
    eigenvalues = eigenvalues[1:, :].astype(complex)[0]
    abs_evs = np.abs(eigenvalues)

    if with_residuals:
        residuals = pd.read_csv(os.path.join(checkpoint_path, 'data', 'residuals.csv')).values[0]
        return eigenvalues, abs_evs, residuals

    else:
        return eigenvalues, abs_evs, None


def get_ev_period(_lambda, temporal_resolution=0.2):
    # period in timesteps: (2*np.pi)/np.abs(lambda_angle)
    # may return inf for non-periodic eigenvalues
    theta = np.angle(_lambda)
    if theta == 0:
        return np.inf
    else:
        return (2 * np.pi) / np.abs(theta) * temporal_resolution


## helper functions
def get_closest_ev_pairs(eigenvalues_dict, n=KEEP_N):
    e1, e2 = eigenvalues_dict.values()

    # e1 and e2 are assumed to be two sets of complex eigenvalues
    s1 = np.vstack([np.real(e1), np.imag(e1)])
    s2 = np.vstack([np.real(e2), np.imag(e2)])
    d = cdist(s1.T, s2.T)

    # matches eigenvalues such that the global! sum of pairwise distances is minimized
    assignment = linear_sum_assignment(d)
    assert n != 0
    if n > 0:
        pair_index = np.argsort(d[assignment])[:n]
    else:  # n < 0
        pair_index = np.argsort(d[assignment])[n:]

    return np.array([assignment[0][pair_index], assignment[1][pair_index]])


def find_and_plot_closest_pairs(cluster_experiments, n_pairs, epsilon=0.0001):
    eigenvalues = {}
    eigenvalue_index = {}
    for exp in cluster_experiments:
        print(f'Getting eigenvalues for {exp}')
        all_eigenvalues, _, residuals = _get_eigenvalues_from_csv(exp, with_residuals=True)
        eigenvalues[exp], eigenvalue_index[exp] = filter_eigenvalues_by_residual(all_eigenvalues, residuals, KEEP_N)

    if n_pairs is not None and len(eigenvalues) == 2:
        # print the indices with the closest distance

        # get the clostest pairs
        clostest_pairs = get_closest_ev_pairs(eigenvalues, n_pairs)
        exp0, exp1 = cluster_experiments
        print(f'Run 1 : {exp0} and Run 2: {exp1}')
        for idx0, idx1 in clostest_pairs.T:
            # want to print the index from before the filtering!
            org_index0, org_index1 = eigenvalue_index[exp0][idx0], eigenvalue_index[exp1][idx1]
            print(
                f'Pair ({idx0}, {idx1}) with $\\lambda_{{{org_index0 + 1}}} = {eigenvalues[exp0][idx0]:.5f}$ ',
                f'and $\\lambda_{{{org_index1 + 1}}} =  {eigenvalues[exp1][idx1]:.5f}$',
            )

        for idx0, _ in clostest_pairs.T:
            org_index0 = eigenvalue_index[exp0][idx0]
            s_per_cycle = get_ev_period(eigenvalues[exp0][idx0])
            if s_per_cycle < np.inf:
                print(f'Period for $\\lambda_{{{org_index0 + 1}}}$: {s_per_cycle:.2f}s/cycle')

        # only keep these for scattering later
        eigenvalues[exp0] = eigenvalues[exp0][clostest_pairs[0]]
        eigenvalues[exp1] = eigenvalues[exp1][clostest_pairs[1]]

    # create the plot
    plt.figure(figsize=(3, 5))
    # add the unit circle
    theta = np.linspace(0, 2 * np.pi, 100)
    plt.plot(np.cos(theta), np.sin(theta), color='black', linewidth=1, label='Unit Circle')

    for exp, evs in eigenvalues.items():
        plt.scatter(x=evs.real, y=evs.imag, s=10, zorder=5, label=f'{exp}')

    plt.xlim(0.8, 1.1)
    plt.ylim(-0.4, 0.4)
    # plt.xlim(.9, 1.03)
    # plt.ylim(-.1, .1)
    plt.grid()
    plt.tight_layout()
    plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05))  # ,ncol=3)
    plt.title(f'{n_pairs} matched eigenvalue pairs for wasserstein distance ', fontdict={'size': 12, 'wrap': True})

    return eigenvalues, clostest_pairs

In [ ]:
# The registry is written by papers/clustering/batch_run_main.py: one entry per
# experiment, holding the run ids of its preprocessing (step 1) and learning
# (step 3) run. The learning run is the one that fitted the model, so that is
# what is inspected here.
RUN_REGISTRY = PROJECT_ROOT / 'papers' / 'clustering' / 'juelich_run_registry.json'

with open(RUN_REGISTRY) as f:
    registry = json.load(f)

In [ ]:
# get an overview over the number of unstable and long-term eigenvalues per run
experiments = [
    '2006bottleneck',
    '2009corner',
    '2009tjunction',
    '2009unidirClosed',
    '2009unidirOpenBs',
    '2009unidirOpenC',
    '2013bidirectionalA',
    '2013bidirectionalB',
    '2013crossing120A',
    '2013crossing120B',
    '2013crossing120C',
    '2013crossing90A',
    '2013crossing90B',
    '2013crossing90C',
    '2013crossing90D',
    '2013crossing90E',
    '2013crossing90F',
    '2013crossing90G',
    '2013singleFile',
    '2013unidirectionalB',
    '2013unidirectionalC',
    '2017simeB',
    '2017simeC',
    '2022wheelchairB',
    '2022wheelchairU',
]

summary_rows = []

for experiment in experiments:
    eigenvalues, abs_evs, _ = _get_eigenvalues_from_csv(experiment, with_residuals=False)

    index_long_term = np.where((abs_evs > 0.999) & (abs_evs < 1.001))
    index_unstable = np.where(abs_evs >= 1.001)

    summary_rows.append(
        {
            'scenario': experiment,
            'unstable': len(eigenvalues[index_unstable]),
            'long_term': len(eigenvalues[index_long_term]),
        }
    )

summary_df = pd.DataFrame(summary_rows).set_index('scenario')
summary_df.columns = ['# unstable', '# long-term']
summary_df

In [ ]:
# get more details on the eigenvalues of the bottleneck scenarios
experiments = [
    '2006bottleneck',
    '2009unidirOpenBs',
    '2013unidirectionalB',
    '2017simeB',
    '2022wheelchairB',
    '2022wheelchairU',
]

for experiment in experiments:
    eigenvalues, abs_evs, residuals = _get_eigenvalues_from_csv(experiment, with_residuals=True)

    filtered_eigenvalues, filtered_index = filter_eigenvalues_by_residual(eigenvalues, residuals, KEEP_N)

    abs_evs = abs_evs[filtered_index]

    index_long_term = np.where((abs_evs >= 0.999) & (abs_evs <= 1.001))[0]
    index_unstable = np.where(abs_evs >= 1.001)[0]
    index_real = np.where(np.abs(filtered_eigenvalues.imag) <= 0.001)[0]  # abs to not get complex conjugate pair

    long_term_evs = filtered_eigenvalues[index_long_term]
    real_evs = filtered_eigenvalues[index_real]
    cycles = [round(get_ev_period(_lambda), 1) for _lambda in long_term_evs]

    print(f'\n======= {experiment} (top {KEEP_N} indices: {filtered_index})')
    print(
        f'long-term: {len(long_term_evs)} at {filtered_index[index_long_term]} -> {np.round(long_term_evs, 4)}, cycles(s)={cycles}'
    )
    print(f'unstable:  {len(index_unstable)}, max|ev|={max(abs_evs):.4f}')
    print(f'real:      {len(real_evs)} at {filtered_index[index_real]} -> {np.round(real_evs, 4)}')

## Plot the eigenvalues for pairs in the clustering 
We want to view the eigenvalues manually and select the ones which are close to one another to then from there identify the modes of interest. 
(Q) What is the spatial pattern with the eigenvalues that are the same across these two scenarios? 


In [ ]:
## plot the eigenvalues for clusters
cluster_experiments = ['2013crossing120A', '2013bidirectionalB']  # '2013bidirectionalB' 2013crossing120A

n_pairs = 25
find_and_plot_closest_pairs(cluster_experiments, n_pairs, epsilon=0.001)

# 2013crossing120C
# plt.scatter(0.99342, 0.00518, s=40, zorder=5, marker='x', c='C0') # \lambda_{3} = 0.99342+0.00518j
plt.scatter(0.97354, +0.02009, s=40, zorder=5, marker='x', c='C0')  # \lambda_{11} = 0.97761-0.01956j

# 2013bidirectionalB
# plt.scatter(0.99407, 0.00206, s=60, zorder=5, marker='x', c='C1') # \lambda_{2} =  0.99407+0.00206j
plt.scatter(0.97433, +0.01848, s=60, zorder=5, marker='x', c='C1')  # \lambda_{9} =  0.97433-0.01848j

plt.xlim(0.85, 1.05)
plt.ylim(-0.08, 0.08)
plt.gca().get_legend().remove()

save_figure('closest_pairs_' + '_'.join(cluster_experiments))
plt.show()

In [ ]:
## plot the eigenvalues for clusters
cluster_experiments = ['2013crossing90D', '2013crossing90E']  # ,

n_pairs = 25
find_and_plot_closest_pairs(cluster_experiments, n_pairs, epsilon=0.001)

# Run 1 : 2013crossing90D and Run 2: 2013crossing90E
# Pair (13, 9) with $\lambda_{28} = 0.92073+0.16460j$  and $\lambda_{10} =  0.96795+0.12166j$
plt.scatter(0.92073, 0.16460, s=40, zorder=5, marker='x', c='C0')  # 2013crossing90D
plt.scatter(0.96795, 0.12166, s=40, zorder=5, marker='x', c='C1')  # 2013bidirectionalB

plt.xlim(0.8, 1.05)
plt.ylim(-0.34, 0.34)
plt.gca().get_legend().remove()

save_figure('closest_pairs_' + '_'.join(cluster_experiments))